# R-MUSAN0: MUSAN 0dB Diversity Intervention -- Colab Cells

## Cell 0: Setup

In [ ]:
# Clone repo + mount Drive
!git clone https://github.com/melodiz/rbpo.git /content/rbpo 2>/dev/null || \
    (cd /content/rbpo && git pull origin main)
%cd /content/rbpo

from google.colab import drive
drive.mount('/content/drive')

## Cell 1: Download checkpoint + BPE model

In [ ]:
import os

MODEL_DIR = '/content/standard_ctc_model'
HF_REPO = 'Zengwei/icefall-asr-librispeech-zipformer-small-ctc-attention-decoder-2024-07-09'

if not os.path.exists(MODEL_DIR):
    !GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/{HF_REPO} {MODEL_DIR}
    !cd {MODEL_DIR} && git lfs pull --include="exp/pretrained.pt"
    !cd {MODEL_DIR} && git lfs pull --include="data/lang_bpe_500/bpe.model"

ckpt_candidates = [
    f'{MODEL_DIR}/exp/pretrained.pt',
    f'{MODEL_DIR}/exp/epoch-30.pt',
    f'{MODEL_DIR}/exp/epoch-50.pt',
]
CKPT_PATH = next((p for p in ckpt_candidates if os.path.exists(p)), None)
if CKPT_PATH is None:
    !ls -la {MODEL_DIR}/exp/
    raise FileNotFoundError(f"No known checkpoint in {MODEL_DIR}/exp/")

BPE_PATH = f'{MODEL_DIR}/data/lang_bpe_500/bpe.model'
assert os.path.exists(BPE_PATH), f"BPE model not found at {BPE_PATH}"

print(f"Checkpoint: {CKPT_PATH}")
print(f"BPE model:  {BPE_PATH}")

## Cell 2: Check existing MUSAN data

In [ ]:
data_dir = "/content/drive/MyDrive/rbpo_results"
musan_dir = f"{data_dir}/musan_rerun"

print("=== Existing MUSAN data ===")
for f in ["cuts_0dB.jsonl.gz", "nbest_0dB_g16.jsonl", "nbest_0dB_g16_pll.jsonl"]:
    path = os.path.join(musan_dir, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    print(f"  {'OK' if exists else 'MISSING'} {f} ({size:.1f} MB)")

AUGMENTED_CUTS = f"{musan_dir}/cuts_0dB.jsonl.gz"
OUTPUT_DIR = f"{data_dir}/R_musan_diversity"

## Cell 3: Install dependencies + icefall

In [ ]:
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")

# Match k2 wheel to Colab's PyTorch + CUDA version
!pip install -q k2==1.24.4.dev20260306+cuda12.8.torch2.10.0 \
    -f https://k2-fsa.github.io/k2/cuda.html

!pip install -q lhotse sentencepiece editdistance transformers

# icefall (provides model architecture code: train.py, get_model, etc.)
import os
if os.path.exists('/content/icefall'):
    !cd /content/icefall && git pull origin master
else:
    !git clone https://github.com/k2-fsa/icefall.git /content/icefall
!cd /content/icefall && pip install -q -e .

ICEFALL_DIR = '/content/icefall'

## Cell 4: Run diversity intervention

In [ ]:
!python /content/rbpo/scripts/musan_diversity_intervention.py \
    --checkpoint {CKPT_PATH} \
    --bpe {BPE_PATH} \
    --icefall-dir {ICEFALL_DIR} \
    --augmented-cuts {AUGMENTED_CUTS} \
    --output-dir {OUTPUT_DIR} \
    --G 16 \
    --tau 10 \
    --device cuda

## Cell 5: View results

In [ ]:
import json

with open(f"{OUTPUT_DIR}/diversity_intervention.json") as f:
    results = json.load(f)

print(f"{'Condition':<20s} {'Oracle':>8s} {'MBR':>8s} {'Delta pp':>8s} "
      f"{'p':>7s} {'Unique':>7s} {'PW-WER':>7s}")
print("-" * 70)
for r in results:
    print(f"{r['condition']:<20s} "
          f"{r['oracle_wer']*100:>7.2f}% "
          f"{r['mbr_wer']*100:>7.2f}% "
          f"{r['delta_vs_greedy_pp']:>+7.3f} "
          f"{r['p_value']:>7.4f} "
          f"{r['mean_unique']:>7.1f} "
          f"{r['mean_pairwise_wer']:>7.4f}")
print(f"\nGreedy WER: {results[0]['greedy_wer']*100:.4f}%")

## Cell 6: Bring-back

In [ ]:
# Copy results to repo for git commit
!mkdir -p /content/rbpo/results/R_musan_diversity/
!cp $OUTPUT_DIR/diversity_intervention.json \
    /content/rbpo/results/R_musan_diversity/
!cp $OUTPUT_DIR/per_condition_diagnostics.csv \
    /content/rbpo/results/R_musan_diversity/

# Large N-best JSONL files stay on Drive (regeneratable)
print("\n=== Bring-back files ===")
!ls -lh /content/rbpo/results/R_musan_diversity/

print("\n=== Stay on Drive (regeneratable) ===")
!ls -lh $OUTPUT_DIR/nbest_*.jsonl 2>/dev/null